# 🎙️➡️🎨 From Voice to Vision — Fase 3: Modelli Baseline

Stabiliamo l'**accuracy di riferimento** prima dell'ottimizzazione:
1. **ML classico** (SVM, Random Forest, Logistic Regression, KNN) sul vettore MFCC.
2. Prima **CNN** sui log-Mel (iperparametri di default, non ancora ottimizzati).

Tutto valutato sullo split **test speaker-independent** (attori mai visti in training).

In [ ]:
# === 1. Repo + librerie + feature in cache ===
REPO_URL = "https://github.com/<tuo-username>/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git","")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

from src import config, data_loader, features
data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)   # carica dalla cache
print("Feature pronte:", data["X_mel"].shape, data["X_vec"].shape)

## 1) Baseline ML classico (vettore MFCC)

In [ ]:
from src.models import baseline
results_ml, scaler = baseline.run_baselines(data)

In [ ]:
# Report dettagliato del miglior modello classico
best_ml = max(results_ml, key=lambda k: results_ml[k]["test_acc"])
print(f"Miglior modello classico: {best_ml}  (test acc = {results_ml[best_ml]['test_acc']:.3f})\n")
print(results_ml[best_ml]["report"])

## 2) Prima CNN sui log-Mel (iperparametri di default)

In [ ]:
from src.models import cnn
import torch
print("Device:", cnn.get_device())

splits = features.split_arrays(data)
hp_default = {"lr": 1e-3, "dropout": 0.3, "weight_decay": 1e-4, "batch_size": 32, "width": 32}
cnn_out = cnn.train_cnn(splits, hp_default, epochs=40, patience=8, verbose=True)

print(f"\nCNN  best_val={cnn_out['best_val_acc']:.3f}  test={cnn_out['test_acc']:.3f}")
print(cnn_out["report"])

In [ ]:
# Curva di training
import matplotlib.pyplot as plt
h = cnn_out["history"]
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(h["train_loss"]); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h["val_acc"]); ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "02_cnn_training.png", dpi=120); plt.show()

## 3) Confronto e matrici di confusione

In [ ]:
import seaborn as sns, numpy as np
def plot_cm(cm, title, ax):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=config.EMOTIONS, yticklabels=config.EMOTIONS, ax=ax)
    ax.set_title(title); ax.set_xlabel("predetto"); ax.set_ylabel("reale")
    ax.tick_params(axis='x', rotation=45); ax.tick_params(axis='y', rotation=0)

fig, ax = plt.subplots(1,2, figsize=(16,6))
plot_cm(results_ml[best_ml]["cm"], f"{best_ml} (test={results_ml[best_ml]['test_acc']:.2f})", ax[0])
plot_cm(cnn_out["cm"], f"CNN (test={cnn_out['test_acc']:.2f})", ax[1])
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "02_confusion_matrices.png", dpi=120); plt.show()

In [ ]:
# Tabella riassuntiva + salvataggio risultati
import pandas as pd, json
rows = [{"model": k, "test_acc": v["test_acc"], "type": "ML"} for k,v in results_ml.items()]
rows.append({"model": "CNN (default)", "test_acc": cnn_out["test_acc"], "type": "Deep"})
table = pd.DataFrame(rows).sort_values("test_acc", ascending=False).reset_index(drop=True)
display(table)

plt.figure(figsize=(9,4))
sns.barplot(data=table, x="test_acc", y="model", hue="type", dodge=False)
plt.xlabel("Test accuracy (speaker-independent)"); plt.title("Baseline: ML classico vs CNN")
plt.xlim(0,1); plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "02_baseline_comparison.png", dpi=120); plt.show()

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "baseline_results.json", "w") as f:
    json.dump({r["model"]: r["test_acc"] for r in rows}, f, indent=2)
print("✓ salvato results/baseline_results.json")

### ✅ Fase 3 completata
Ora abbiamo l'accuracy di riferimento: ML classico vs una CNN non ancora ottimizzata.

**Mandami:** la tabella finale + le figure (training, matrici di confusione, confronto).
In **Fase 4** ottimizzeremo gli iperparametri della CNN con **PSO, Flock of Starlings e GA** a confronto — il cuore "bio-ispirato" del progetto.